## Before you run this

**Training writes to `runs/`, never to `dataset/checkpoints/`.** Every config now sets
`overwrite: false`, so if the target directory already exists the run aborts with
`FileExistsError` instead of silently deleting a published checkpoint. Rename or remove
the directory under `runs/` when you want to re-run.

**`Run All` trains from scratch.** The teacher takes 75 epochs, the student up to 1000
(with early stopping). To only check that the pipeline is wired up, run the cells up to
the first validation and interrupt, or lower `epochs` in the config.

**Hyperparameter search is off** (`USE_OPTUNA = False`). Turning it on re-runs the
Optuna study, which took hours per notebook.


In [ ]:
# ── 133-kp config: MUST run BEFORE the imports (config/skeleton read these env vars) ──
import os

# Keypoint layout: "coco133" = RTMW 133 -> 55-joint signing subset + GCN.
# ("mediapipe75" for the legacy teacher.)
os.environ["KP_LAYOUT"] = "coco133"

# EDIT: directory that DIRECTLY contains Phoenix-2014T.train / .dev / .test
os.environ["MSKA_DATA_DIR"] = "/kaggle/input/phoenix133/phoenix2014t_133kp"

# Parallel loaders on Linux (keeps the GPU busy). Use "0" on Windows.
os.environ["NUM_WORKERS"] = "8"

# ── Where this run writes ──────────────────────────────────────────────────
RUN_NAME  = "run133"
OVERWRITE = False

RUN_DIR = os.path.join("../../runs", RUN_NAME)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    _n = 2
    while os.path.exists(f"{RUN_DIR}_{_n:03d}"):
        _n += 1
    RUN_DIR = f"{RUN_DIR}_{_n:03d}"
print("this run writes to:", os.path.normpath(os.path.abspath(RUN_DIR)))

# "mine" -> the test cell evaluates the freshly trained checkpoint (in RUN_DIR)
EVAL_CKPT = "mine"
TRAIN_CKPT = os.path.join(RUN_DIR, "tssi133_gcn_best.pt")

# CSRL — 75-keypoint Skeleton CSLR Runner

Modular runner notebook. All logic lives in `.py` modules:
- `config.py` — device, paths, CFG
- `skeleton.py` — DFS graph, TSSI generation
- `vocab.py` — vocabulary with gloss merge
- `dataset.py` — TSSIDataset, collate, dataloaders
- `model.py` — PoseNetworkCTC
- `losses.py` — CTC loss, decoding, WER
- `training.py` — two-phase training with resume checkpoints
- `optuna_search.py` — hyperparameter search

## 0 — Imports and setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from config import DEVICE, AMP, DATA_DIR, CKPT_PATH, CFG
from vocab import build_vocab_from_raw
from dataset import load_pkl, make_dataloaders
from model import PoseNetworkCTC
from training import run_training, evaluate

print(f"device: {DEVICE} | amp: {AMP}")
print(f"layout/joints -> in_channels: {CFG['num_joints']} joints, use_gcn={CFG['use_gcn']}")
print(f"CFG: {CFG}")

## 1 — Load data and build vocabulary

In [ ]:
import os

train_raw = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.train"))
dev_raw   = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.dev"))
test_raw  = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.test"))

print("Building vocabulary...")
V = build_vocab_from_raw(train_raw, dev_raw, test_raw)
NUM_CLASSES = V["num_classes"]
LOG_PRIOR = V["log_prior"].to(DEVICE)
i2g = V["i2g"]

train_loader, dev_loader, test_loader, _, _, _ = make_dataloaders(
    DATA_DIR, V["gloss_to_ids"], CFG
)
print(f"vocab: {NUM_CLASSES} classes")   # atteso ~1087 (1085 gloss + blank + unk)

## 2 — Instantiate model

In [ ]:
model = PoseNetworkCTC(
    num_classes=NUM_CLASSES,
    in_channels=CFG["num_joints"] * 3,
    hidden_dim=CFG["hidden_dim"],
    tcn_blocks=CFG["tcn_blocks"],
    lstm_layers=CFG["num_layers"],
    dropout=CFG["dropout"],
    drop_path_rate=CFG["drop_path_rate"],
    attn_heads=CFG["attn_heads"],
    use_gcn=CFG["use_gcn"],
    gcn_channels=CFG["gcn_channels"],
).to(DEVICE)

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

## 3 — (Optional) Optuna hyperparameter search

Set `USE_OPTUNA = True` to run. Trials live in `optuna_cslr.db` (SQLite), one study
per experiment (auto-named from the keypoint layout, e.g. `cslr_coco133_gcn1`) —
resume is automatic and different experiments never mix. Median pruning kills bad
trials after ~2 epochs of phase 2, so a 12-trial budget costs far less than 12
full screenings.

In [ ]:
USE_OPTUNA = False   # set True to run the search (SQLite storage + median pruning)

if USE_OPTUNA:
    from optuna_search import run_optuna_search

    # Screening over the first 28 epochs of the real schedule, with MEDIAN PRUNING:
    # poor trials die after ~2 epochs of Phase 2, so n_trials can be
    # generous (pruned trials are cheap). The study lives in optuna_cslr.db
    # with an auto per-experiment name (e.g. "cslr_coco133_gcn1"): automatic resume
    # and NO contamination across different layouts/models.
    best_params = run_optuna_search(
        cfg=CFG,
        train_loader=train_loader,
        dev_loader=dev_loader,
        num_classes=NUM_CLASSES,
        log_prior=LOG_PRIOR,
        device=DEVICE,
        amp=AMP,                       # remember: AMP=False must be set BEFORE this cell
        n_trials=12,
        trial_train_epochs=28,
        storage="sqlite:///optuna_cslr.db",
    )
    for k, v in best_params.items():
        CFG[k] = v
    if best_params:
        CFG["phase2_lr_backbone"] = CFG["phase2_lr_head"] * 0.5
        print("CFG updated with Optuna best.")
else:
    print("Optuna search disabled — using current CFG values.")

# always rebuild model with fresh weights (Optuna may have left dirty weights in memory)
model = PoseNetworkCTC(
    num_classes=NUM_CLASSES,
    in_channels=CFG["num_joints"] * 3,
    hidden_dim=CFG["hidden_dim"],
    tcn_blocks=CFG["tcn_blocks"],
    lstm_layers=CFG["num_layers"],
    dropout=CFG["dropout"],
    drop_path_rate=CFG["drop_path_rate"],
    attn_heads=CFG["attn_heads"],
    use_gcn=CFG["use_gcn"],
    gcn_channels=CFG["gcn_channels"],
).to(DEVICE)
print(f"fresh model: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params")

## 4 — Train (two-phase with resume checkpoints)

In [ ]:
BEST_FALLBACK = ""  # set path to an existing .pt to use as fallback, or "" to ignore

os.makedirs(RUN_DIR, exist_ok=True)
hist, best_wer = run_training(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    cfg=CFG,
    device=DEVICE,
    amp=AMP,
    log_prior=LOG_PRIOR,
    ckpt_path=TRAIN_CKPT,
    best_fallback_path=BEST_FALLBACK,
    fresh_start=True,  # set False to resume from checkpoint
)

## 5 — Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(hist["ep"], hist["loss"], marker="o", ms=3, label="train (CTC + SR-CTC)")
if hist.get("val_loss") and any(v == v for v in hist["val_loss"]):  # any non-NaN
    ax1.plot(hist["ep"], hist["val_loss"], marker="o", ms=3,
             color="tab:orange", label="dev (main CTC)")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.grid(alpha=0.3)
ax1.legend()

if hist["wer"]:
    bi = int(np.argmin(hist["wer"]))
    ax2.plot(hist["ep"], hist["wer"], marker="o", ms=3, color="crimson")
    ax2.scatter([hist["ep"][bi]], [hist["wer"][bi]], color="k", zorder=5,
               label=f"best {hist['wer'][bi]:.2f}%")
    if 2 in hist["phase"]:
        f2 = hist["ep"][hist["phase"].index(2)]
        ax1.axvline(f2, ls="--", c="gray", alpha=0.6)
        ax2.axvline(f2, ls="--", c="gray", alpha=0.6, label="Phase 2 start")
    ax2.legend()

ax2.set_title("Dev WER (corpus-level)")
ax2.set_xlabel("epoch")
ax2.set_ylabel("WER %")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("tssi75_curves.png", dpi=110)
plt.show()

## 6 — Test set evaluation (greedy vs beam)

In [ ]:
_eval_path = CKPT_PATH if EVAL_CKPT == "published" else TRAIN_CKPT
print("evaluating:", _eval_path)
ck = torch.load(_eval_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
model.to(DEVICE).eval()
print(f"checkpoint epoch {ck['epoch']} | dev WER {ck['wer']*100:.2f}%\n")

wer_g, dg, _ = evaluate(model, test_loader, CFG, DEVICE, LOG_PRIOR, use_beam=False)
print(f"TEST greedy      WER {wer_g*100:.2f}%  (S{dg['S']} D{dg['D']} I{dg['I']} N{dg['N']})")

wer_b, db, _ = evaluate(model, test_loader, CFG, DEVICE, LOG_PRIOR, use_beam=True)
print(f"TEST beam({CFG['beam_width']})    WER {wer_b*100:.2f}%  (S{db['S']} D{db['D']} I{db['I']} N{db['N']})")

print(f"\n>>> WER to report = {min(wer_g, wer_b)*100:.2f}%")